In [2]:
import pandas as pd
import duckdb

### Предметная область

Обезличенные данные о клиентах банка. Одна строка - один клиент.

Удержать клиента дешевле, чем привлечь нового, поэтому цель анализа - выявить тех, кто уходит, и предложить решение этой проблемы.

Для начала работы загрузим датасет

In [3]:
df = pd.read_csv("../data/Customer-Churn-Records.csv")
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0,5,GOLD,350
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0,5,GOLD,425


In [4]:
df.shape

(10000, 18)

Размер датасета - 10 000 строк и 18 столбцов.

### Описание полей

In [5]:
df.dtypes

RowNumber               int64
CustomerId              int64
Surname                   str
CreditScore             int64
Geography                 str
Gender                    str
Age                     int64
Tenure                  int64
Balance               float64
NumOfProducts           int64
HasCrCard               int64
IsActiveMember          int64
EstimatedSalary       float64
Exited                  int64
Complain                int64
Satisfaction Score      int64
Card Type                 str
Point Earned            int64
dtype: object



| Столбец | Описание |
|---|---|
| `CreditScore` | Кредитный рейтинг (чем выше, тем надёжнее клиент) |
| `Geography` | Страна (Франция, Германия, Испания) |
| `Gender` | Пол |
| `Age` | Возраст |
| `Tenure` | Сколько лет клиент с банком |
| `Balance` | Остаток на счёте |
| `NumOfProducts` | Сколько продуктов банка у клиента (карта, вклад, кредит…) |
| `HasCrCard` | Есть ли кредитная карта (1 - да, 0 - нет) |
| `IsActiveMember` | Активно ли клиент пользуется банком (1 - да, 0 - нет) |
| `EstimatedSalary` | Примерная зарплата |
| `Complain` | Была ли жалоба (1 - да, 0 - нет) |
| `Satisfaction Score` | Оценка удовлетворённости (1-5) |
| `Card Type` | Тип карты (Silver, Gold, Platinum, Diamond) |
| `Point Earned` | Баллы программы лояльности |
| `Exited` | **Ушёл ли клиент (1 - ушёл, 0 - остался) - целевая переменная** |

In [6]:
df['Exited'].value_counts(normalize=True)

Exited
0    0.7962
1    0.2038
Name: proportion, dtype: float64

Из банка ушло 20% клиентов. Классы несбалансированы, это нужно учитывать при сравнении групп.

In [7]:
df['Geography'].value_counts(normalize=True)

Geography
France     0.5014
Germany    0.2509
Spain      0.2477
Name: proportion, dtype: float64

Половина клиентов из Франции, остальная половина из Германии и Испании.

In [8]:
df.isna().sum().sum()

np.int64(0)

В датасете нет пропусков.

In [9]:
df.duplicated().sum()

np.int64(0)

В датасете нет дубликатов.

In [10]:
df = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'])

Удалили неинформативные столбцы.

In [11]:
df.describe()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Point Earned
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,650.528800,38.921800,5.012800,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203800,0.204400,3.013800,606.515100
std,96.653299,10.487806,2.892174,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402842,0.403283,1.405919,225.924839
min,350.000000,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000,0.000000,1.000000,119.000000
25%,584.000000,32.000000,3.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000,0.000000,2.000000,410.000000
50%,652.000000,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000,0.000000,3.000000,605.000000
75%,718.000000,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000,0.000000,4.000000,801.000000
max,850.000000,92.000000,10.000000,250898.090000,4.000000,1.00000,1.000000,199992.480000,1.000000,1.000000,5.000000,1000.000000


Возраст клиентов от 18 до 92 лет, медиана - 37 лет.

Не менее 25% клиентов имеют нулевые балансы, возможно, это «спящие» счета, поэтому проверим их отток отдельно.

Большинство клиентов имеют кредитную карту.

Большинство клиентов имеют 1-2 продукта банка.

Активных клиентов - 51%.

20% клиентов оставили жалобу на банк.

Средняя оценка удовлетворённости банком - 3.

Сформируем гипотезы:

**1. Гипотеза:** клиенты с несколькими продуктами уходят реже.\
*Обоснование:* большое количество продуктов формирует привязанность к банку.\
*В случае подтверждения:* предложить использование кросс-продаж.

**2. Гипотеза:** неактивные клиенты уходят чаще.\
*Обоснование:* возможно, они уже готовятся к полному переходу в другой банк.\
*В случае подтверждения:* предложить программы реактивации.

**3. Гипотеза:** чем выше статус карты, тем ниже отток.\
*Обоснование:* премиальные карты дают больше привилегий, которые сжигаются при переходе в другой банк.\
*В случае подтверждения:* предложить улучшить систему апгрейдов карты для удержания клиентов.

**4. Гипотеза:** отток различается у мужчин и женщин.\
*Обоснование:* проверим, есть ли различия в поведении сегментов.\
*В случае подтверждения:* учитывать пол при сегментации клиентов.

**5. Гипотеза:** клиенты младше 37 лет (медианный возраст) уходят чаще.\
*Обоснование:* молодым легче переключаться между банками.\
*В случае подтверждения:* предложить таргетированные по возрасту предложения.

**6. Гипотеза:** клиенты, которые удовлетворены банком на 1 или 2 балла, уходят чаще.\
*Обоснование:* клиенты уходят из-за недовольства банком.\
*В случае подтверждения:* в первую очередь работать с недовольными клиентами, которые ещё не ушли.

### SQL

Гипотеза 1

In [13]:
query = """
SELECT NumOfProducts AS products, COUNT(*) AS clients, ROUND(AVG(Exited) * 100, 2) AS churn_percent
FROM df
GROUP BY NumOfProducts
ORDER BY NumOfProducts
"""
duckdb.sql(query).df()

,products,clients,churn_percent
0,1,5084,27.71
1,2,4590,7.60
2,3,266,82.71
3,4,60,100.00


Гипотеза 2

In [14]:
query = """
SELECT IsActiveMember AS active, COUNT(*) AS clients, ROUND(AVG(Exited) * 100, 2) AS churn_percent
FROM df
GROUP BY IsActiveMember
ORDER BY IsActiveMember
"""
duckdb.sql(query).df()

,active,clients,churn_percent
0,0,4849,26.87
1,1,5151,14.27


Гипотеза 3

In [32]:
query = """
SELECT "Card Type" AS card_type, COUNT(*) AS clients, ROUND(AVG(Exited) * 100, 2) AS churn_percent
FROM df
GROUP BY card_type
ORDER BY CASE WHEN card_type = 'SILVER' THEN 1
              WHEN card_type = 'GOLD' THEN 2
              WHEN card_type = 'PLATINUM' THEN 3
              WHEN card_type = 'DIAMOND' THEN 4 END
"""
duckdb.sql(query).df()

,card_type,clients,churn_percent
0,SILVER,2496,20.11
1,GOLD,2502,19.26
2,PLATINUM,2495,20.36
3,DIAMOND,2507,21.78


Гипотеза 4

In [61]:
query = """
SELECT Gender AS gender, COUNT(*) AS clients, ROUND(AVG(Exited) * 100, 2) AS churn_percent
FROM df
GROUP BY gender
ORDER BY churn_percent
"""
duckdb.sql(query).df()


,gender,clients,churn_percent
0,Male,5457,16.47
1,Female,4543,25.07


Гипотеза 5

In [31]:
query = """
SELECT COUNT(*) AS clients, MIN(Age) AS min_age, MAX(Age) AS max_age, ROUND(AVG(Exited) * 100, 2) AS churn_percent
FROM df
WHERE age < 37
UNION ALL
SELECT COUNT(*) AS clients, MIN(Age) AS min_age, MAX(Age) AS max_age, ROUND(AVG(Exited) * 100, 2) AS churn_percent
FROM df
WHERE age >= 37
ORDER BY min_age
"""
duckdb.sql(query).df()

,clients,min_age,max_age,churn_percent
0,4609,18,36,8.68
1,5391,37,92,30.38


In [60]:
query = """
SELECT CASE WHEN Age <= 27 THEN '18 - 27'
            WHEN Age <= 37 THEN '28 - 37'
            WHEN Age <= 47 THEN '38 - 47'
            WHEN Age <= 57 THEN '48 - 57'
            ELSE '58+' END AS age_group,
       COUNT(*) AS clients, ROUND(AVG(Exited) * 100, 2) AS churn_percent
FROM df
GROUP BY age_group
ORDER BY age_group
"""
duckdb.sql(query).df()

,age_group,clients,churn_percent
0,18 - 27,1020,7.16
1,28 - 37,4067,9.56
2,38 - 47,3203,24.57
3,48 - 57,1055,55.26
4,58+,655,31.45


Гипотеза 6

In [25]:
query = """
SELECT "Satisfaction Score" AS satisfaction, COUNT(*) AS clients, ROUND(AVG(Exited) * 100, 2) AS churn_percent
FROM df
GROUP BY satisfaction
ORDER BY satisfaction
"""
duckdb.sql(query).df()

,satisfaction,clients,churn_percent
0,1,1932,20.03
1,2,2014,21.80
2,3,2042,19.64
3,4,2008,20.62
4,5,2004,19.81


In [35]:
query = """
SELECT CASE WHEN "Satisfaction Score" BETWEEN 1 AND 2 THEN '1 - 2' ELSE '3 - 5' END AS satisfaction, COUNT(*) AS clients, ROUND(AVG(Exited) * 100, 2) AS churn_percent
FROM df
GROUP BY satisfaction
ORDER BY satisfaction
"""
duckdb.sql(query).df()

,satisfaction,clients,churn_percent
0,1-2,3946,20.93
1,3-5,6054,20.02


Спящие счета

In [63]:
query = """
SELECT Balance AS balance, COUNT(*) AS clients, ROUND(AVG(IsActiveMember) * 100, 2) AS active_percent
FROM df
WHERE Balance = 0
GROUP BY Balance
"""
duckdb.sql(query).df()

,balance,clients,active_percent
0,0.0,3617,51.78


In [57]:
query = """
SELECT Balance AS balance, COUNT(*) AS clients, ROUND(AVG(Exited) * 100, 2) AS churn_percent
FROM df
WHERE Balance = 0 AND IsActiveMember = 0
GROUP BY Balance
"""

duckdb.sql(query).df()

,balance,clients,churn_percent
0,0.0,1744,18.35


Сегментирование

In [56]:
query = """
SELECT IsActiveMember AS active, Gender As gender, COUNT(*) AS clients, ROUND(AVG(Exited) * 100, 2) AS churn_percent
FROM df
GROUP BY active, gender
ORDER BY churn_percent
"""
duckdb.sql(query).df()

,active,gender,clients,churn_percent
0,1,Male,2867,11.20
1,1,Female,2284,18.13
2,0,Male,2590,22.32
3,0,Female,2259,32.09


In [55]:
query = """
WITH segments AS(
SELECT CASE WHEN IsActiveMember = 1 AND Gender = 'Male' AND Age > 37 THEN 'old_active_male' WHEN IsActiveMember = 1 AND Gender = 'Male' AND Age <= 37 THEN 'young_active_male' WHEN IsActiveMember = 0 AND Gender = 'Male' AND Age > 37 THEN 'old_not_active_male' WHEN IsActiveMember = 0 AND Gender = 'Male' AND Age <= 37 THEN 'young_not_active_male' WHEN IsActiveMember = 1 AND Gender = 'Female' AND Age > 37 THEN 'old_active_female' WHEN IsActiveMember = 0 AND Gender = 'Female' AND Age > 37 THEN 'old_not_active_female' WHEN IsActiveMember = 1 AND Gender = 'Female' AND Age <= 37 THEN 'young_active_female' WHEN IsActiveMember = 0 AND Gender = 'Female' AND Age <= 37 THEN 'young_not_active_female' END AS segment, Exited
FROM df)

SELECT segment, COUNT(*) AS clients, ROUND(AVG(Exited) * 100, 2) AS churn_percent
FROM segments
GROUP BY segment
ORDER BY churn_percent
"""

duckdb.sql(query).df()

,segment,clients,churn_percent
0,young_active_male,1467,4.91
1,young_not_active_male,1394,9.04
2,young_active_female,1129,9.21
3,young_not_active_female,1097,14.59
4,old_active_male,1400,17.79
5,old_active_female,1155,26.84
6,old_not_active_male,1196,37.79
7,old_not_active_female,1162,48.62
